# CMPE 401 Project 1: Colab runner

Runs one experiment from `configs/experiments.yaml` end to end: train, evaluate, and plot.

1. **Runtime > Change runtime type > GPU** (T4 or better).
2. Set `RUN` below and run all cells.
3. If Colab disconnects, reconnect and **run all again**. Training resumes from the last epoch saved on Google Drive.
4. When it finishes, download `MyDrive/cmpe401/results/<RUN>/` and add it to the repo's `results/` folder.

In [ ]:
RUN = "p3_lr02"    # done: baseline | next: p3_lr02, p3_lr005 | later: p4_cos_lr, p4_regularize, p3_imgsz960
SMOKE_TEST = False # True = 1 epoch on 5% of the data, to check the pipeline before a long run

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE = "/content/drive/MyDrive/cmpe401"
os.environ["RUNS_DIR"] = f"{DRIVE}/runs"        # weights + checkpoints (survive disconnects)
os.environ["RESULTS_DIR"] = f"{DRIVE}/results"  # CSVs, plots, metrics to commit to the repo
os.makedirs(os.environ["RUNS_DIR"], exist_ok=True)
os.makedirs(os.environ["RESULTS_DIR"], exist_ok=True)

In [ ]:
%cd /content
!test -d repo || git clone https://github.com/EzraKrause04/CMPE-401---Instructor-Defined-Project-1.git repo
%cd /content/repo
!git pull -q
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# VisDrone (~2 GB) downloads to fast local disk on first use in each session (a few minutes).
!yolo settings datasets_dir=/content/datasets

In [ ]:
if SMOKE_TEST:
    !python src/train.py {RUN} --epochs 1 --fraction 0.05
else:
    !python src/train.py {RUN}

In [ ]:
if not SMOKE_TEST:
    !python src/evaluate.py {RUN} --split val
    !python src/evaluate.py {RUN} --split test
    !python src/plot_curves.py {RUN}

In [ ]:
from IPython.display import Image, display
if not SMOKE_TEST:
    display(Image(f"{os.environ['RESULTS_DIR']}/{RUN}/loss_curves.png"))